# Synthetic → Tournament transfer demo (binary cheating detector)

**Goal:** train a simple detector on **synthetic** move embeddings, then evaluate the same model on a **real tournament** dataset (zero-shot transfer).

What we do:
1. Load synthetic dataset metadata (CSV) + **Allie embeddings stored as `*.npy`** (one file per move column).
2. Build a binary dataset:  
   - **y=0**: played move (`move_uci`)  
   - **y=1**: injected cheat move (default: `move_stockfish_15`)
3. Train **logistic regression** = a single linear layer (`nn.Linear`) using SGD.
4. Pick a decision threshold on a small validation split.
5. Report metrics on:
   - synthetic TEST
   - tournament dataset (transfer)

## Setup

In [26]:
!pip -q install pytorch-lightning

In [27]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
# === Imports ===
from __future__ import annotations

import gc
import random
from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

import pytorch_lightning as pl

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)


In [29]:
# === Reproducibility ===
SEED = 42
pl.seed_everything(SEED, workers=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)


INFO:lightning_fabric.utilities.seed:Seed set to 42


DEVICE: cuda


## Paths

Edit these four paths to match your Colab environment.

In [30]:
# -------------------------
# Synthetic (train)
# -------------------------
SYNTH_CSV_PATH = "/content/drive/MyDrive/Skoltech_chess_project/chess-fraud-synth/chess_project_pov_synth_dataset.csv"
SYNTH_EMB_HUMAN = "/content/drive/MyDrive/Skoltech_chess_project/chess-fraud-synth/embs_allie_2500_small_fp16/move_uci.npy"
SYNTH_EMB_CHEAT = "/content/drive/MyDrive/Skoltech_chess_project/chess-fraud-synth/embs_allie_2500_small_fp16/move_stockfish_15.npy"

# Which synthetic "cheat move" to treat as positive class.
CHEAT_MOVE_COL = "move_stockfish_15"

# -------------------------
# Tournament (transfer eval)
# -------------------------
TOUR_CSV_PATH = "/content/drive/MyDrive/Skoltech_chess_project/chess-fraud-tournament/skoltech_project_pov_tournament_dataset"
TOUR_EMB_NPZ = "/content/drive/MyDrive/Skoltech_chess_project/chess-fraud-tournament/embs_allie_2500.npz"
TOUR_EMB_KEY = "move_uci"  # key inside the tournament NPZ


In [31]:
# Quick path checks
for p in [SYNTH_CSV_PATH, SYNTH_EMB_HUMAN, SYNTH_EMB_CHEAT, TOUR_CSV_PATH, TOUR_EMB_NPZ]:
    print(p, "->", "OK" if Path(p).exists() else "MISSING")

/content/drive/MyDrive/Skoltech_chess_project/chess-fraud-synth/chess_project_pov_synth_dataset.csv -> OK
/content/drive/MyDrive/Skoltech_chess_project/chess-fraud-synth/embs_allie_2500_small_fp16/move_uci.npy -> OK
/content/drive/MyDrive/Skoltech_chess_project/chess-fraud-synth/embs_allie_2500_small_fp16/move_stockfish_15.npy -> OK
/content/drive/MyDrive/Skoltech_chess_project/chess-fraud-tournament/skoltech_project_pov_tournament_dataset -> OK
/content/drive/MyDrive/Skoltech_chess_project/chess-fraud-tournament/embs_allie_2500.npz -> OK


## Load synthetic CSV and `*.npy` embeddings


Each file is an array of shape **(N, D)** aligned with the **rows of the synthetic CSV**.

In [32]:
def load_npy_matrix(path: str | Path) -> np.ndarray:
    path = Path(path)
    return np.load(path, mmap_mode="r")

def describe_matrix(name: str, x: np.ndarray) -> None:
    print(f"{name}: shape={tuple(x.shape)}, dtype={x.dtype}")


In [33]:
# --- Load CSV ---
synth_df = pd.read_csv(
    SYNTH_CSV_PATH,
    usecols=["split_by_player", "rating_bin", "player"],
)
print("split_by_player counts:", synth_df["split_by_player"].value_counts().to_dict())


split_by_player counts: {'train': 334140, 'test': 83067}


In [34]:
# --- Load embeddings from *.npy (aligned with the CSV rows) ---
human_emb = load_npy_matrix(SYNTH_EMB_HUMAN)  # np.float16 memmap
cheat_emb = load_npy_matrix(SYNTH_EMB_CHEAT)  # np.float16 memmap

describe_matrix("human_emb_full", human_emb)
describe_matrix("cheat_emb_full", cheat_emb)

if human_emb.shape[0] != len(synth_df) or cheat_emb.shape[0] != len(synth_df):
    raise ValueError("Embeddings row count does not match CSV row count")
if human_emb.shape[1] != cheat_emb.shape[1]:
    raise ValueError("human_emb and cheat_emb must have the same embedding dim")


human_emb_full: shape=(417207, 1024), dtype=float16
cheat_emb_full: shape=(417207, 1024), dtype=float16


## Train/val/test split (player-level, inside each rating bin)

- We use the existing `split_by_player` column for **train vs test** on synthetic.
- Inside synthetic TRAIN we create a small **validation set** by holding out a fraction of **players per rating bin**.

This keeps the split logic conceptually similar to the “real” pipeline, but still easy to follow.

In [35]:
# Stable bin index (0..K-1) for stratified per-bin operations
bin_names = sorted(synth_df["rating_bin"].unique().tolist())
bin_to_idx = {b: i for i, b in enumerate(bin_names)}
synth_df["bin_idx"] = synth_df["rating_bin"].map(bin_to_idx)

print("Bins:", bin_names)

Bins: ['1200-1400', '1400-1600', '1600-1800', '1800-2000', '2000-2200', 'lt_1200']


In [36]:
def split_train_val_by_player_per_bin(
    df: pd.DataFrame,
    *,
    split_col: str,
    bin_idx_col: str,
    player_col: str,
    val_frac: float,
    seed: int,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    split = df[split_col].to_numpy()
    is_train = (split == "train")
    is_test = (split == "test")
    bin_idx = df[bin_idx_col].to_numpy()

    rng = np.random.default_rng(seed)
    train_idx = []
    val_idx = []

    for b in np.unique(bin_idx):
        rows = np.flatnonzero(is_train & (bin_idx == b))
        players = df.loc[rows, player_col].to_numpy()
        uniq_players = np.unique(players)

        n_val_players = max(1, int(round(val_frac * len(uniq_players))))
        val_players = uniq_players[rng.permutation(len(uniq_players))[:n_val_players]]

        is_val = np.isin(players, val_players)
        val_idx.append(rows[is_val])
        train_idx.append(rows[~is_val])

    train_idx = np.concatenate(train_idx) if train_idx else np.array([], dtype=int)
    val_idx = np.concatenate(val_idx) if val_idx else np.array([], dtype=int)
    test_idx = np.flatnonzero(is_test)

    return train_idx, val_idx, test_idx

In [37]:
train_idx, val_idx, test_idx = split_train_val_by_player_per_bin(
    synth_df,
    split_col="split_by_player",
    bin_idx_col="bin_idx",
    player_col="player",
    val_frac=0.10,
    seed=SEED,
)

print({"train": len(train_idx), "val": len(val_idx), "test": len(test_idx)})

# free the big CSV frame (we only need the split indices further)
del synth_df
gc.collect()


{'train': 301304, 'val': 32836, 'test': 83067}


0

## Build a binary dataset from embeddings

For each eligible position (row in `df_f`) we create **two samples**:

- sample A: `emb_human[i]`, label `0`
- sample B: `emb_cheat[i]`, label `1`

So if there are `N` eligible rows, the classifier sees `2N` training examples.

In [38]:
class RowPairDataset(Dataset):
    """Two-class dataset built from the same row indices:
    - class 0: human_emb[row]
    - class 1: cheat_emb[row]
    Ordering matches the old concatenation: [all human] + [all cheat].
    """

    def __init__(self, human_emb: np.ndarray, cheat_emb: np.ndarray, row_idx: np.ndarray):
        self.human_emb = human_emb
        self.cheat_emb = cheat_emb
        self.row_idx = np.asarray(row_idx, dtype=np.int64)

    def __len__(self) -> int:
        return int(2 * len(self.row_idx))

    def __getitem__(self, idx: int):
        n = len(self.row_idx)
        if idx < n:
            row = int(self.row_idx[idx])
            x = self.human_emb[row]
            y = np.float32(0.0)
        else:
            row = int(self.row_idx[idx - n])
            x = self.cheat_emb[row]
            y = np.float32(1.0)
        return x, y


def build_y(row_idx: np.ndarray) -> np.ndarray:
    n = int(len(row_idx))
    return np.concatenate(
        [np.zeros((n,), dtype=np.float32), np.ones((n,), dtype=np.float32)],
        axis=0,
    )


In [39]:
train_dataset = RowPairDataset(human_emb, cheat_emb, train_idx)
val_dataset = RowPairDataset(human_emb, cheat_emb, val_idx)
test_dataset = RowPairDataset(human_emb, cheat_emb, test_idx)

y_train = build_y(train_idx)
y_val = build_y(val_idx)
y_test = build_y(test_idx)

print("Dataset sizes (samples):")
print("  train_dataset", len(train_dataset), "y_train", y_train.shape)
print("  val_dataset  ", len(val_dataset), "y_val  ", y_val.shape)
print("  test_dataset ", len(test_dataset), "y_test ", y_test.shape)
print("Embedding dim:", int(human_emb.shape[1]))


Dataset sizes (samples):
  train_dataset 602608 y_train (602608,)
  val_dataset   65672 y_val   (65672,)
  test_dataset  166134 y_test  (166134,)
Embedding dim: 1024


## Model: one linear layer (logistic regression)

We train:

$$
p(y=1\mid x)=\sigma(w^T x + b)
$$

with minibatch Adam using PyTorch Lightnin.


In [40]:
class LogregModule(pl.LightningModule):
    def __init__(self, input_dim: int, *, lr: float = 3e-4, weight_decay: float = 0.0):
        super().__init__()
        self.save_hyperparameters()
        self.linear = nn.Linear(input_dim, 1)
        self.loss_fn = nn.BCEWithLogitsLoss()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.to(dtype=torch.float32)
        return self.linear(x).squeeze(-1)

    def training_step(self, batch: Tuple[torch.Tensor, torch.Tensor], batch_idx: int) -> torch.Tensor:
        features, labels = batch
        logits = self(features)
        loss = self.loss_fn(logits, labels.to(dtype=torch.float32))
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch: Tuple[torch.Tensor, torch.Tensor], batch_idx: int) -> torch.Tensor:
        features, labels = batch
        logits = self(features)
        loss = self.loss_fn(logits, labels.to(dtype=torch.float32))
        self.log("val_loss", loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr, weight_decay=self.hparams.weight_decay)


In [41]:
class XYDataModule(pl.LightningDataModule):
    def __init__(
        self,
        train_dataset: Dataset,
        val_dataset: Dataset,
        *,
        batch_size: int = 1024,
        num_workers: int = 0,
    ):
        super().__init__()
        self.train_dataset = train_dataset
        self.val_dataset = val_dataset
        self.batch_size = batch_size
        self.num_workers = num_workers

    def setup(self, stage: str | None = None) -> None:
        pass

    def train_dataloader(self) -> DataLoader:
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
            pin_memory=(DEVICE.type == "cuda"),
        )

    def val_dataloader(self) -> DataLoader:
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=(DEVICE.type == "cuda"),
        )


In [42]:
batch_size = 1024
epochs = 10
lr = 3e-4
weight_decay = 0.0

In [43]:
data_module = XYDataModule(
    train_dataset,
    val_dataset,
    batch_size=batch_size,
)

model = LogregModule(input_dim=int(human_emb.shape[1]), lr=lr, weight_decay=weight_decay)

trainer = pl.Trainer(
    max_epochs=epochs,
    accelerator="auto",
    devices=1,
    logger=False,
    enable_checkpointing=False,
    enable_model_summary=False,
    log_every_n_steps=50,
)

trainer.fit(model, datamodule=data_module)
model = model.to(DEVICE)


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=10` reached.


## Threshold selection on validation

We choose the threshold that maximizes **F1 for the positive class (cheat)** on the validation set.

In [44]:
@torch.no_grad()
def predict_proba_loader(model: nn.Module, dataloader: DataLoader) -> np.ndarray:
    model.eval()
    device = next(model.parameters()).device

    probs = []
    for features, _labels in dataloader:
        features = features.to(device=device, dtype=torch.float32, non_blocking=True)
        probs.append(torch.sigmoid(model(features)).cpu())

    return torch.cat(probs, dim=0).numpy()


def pick_threshold_by_f1(y_true: np.ndarray, probs: np.ndarray, grid: np.ndarray) -> Tuple[float, float]:
    best_thr = float(grid[0])
    best_f1 = -1.0
    for thr in grid:
        y_hat = (probs >= thr).astype(int)
        f1 = f1_score(y_true, y_hat)
        if f1 > best_f1:
            best_f1 = float(f1)
            best_thr = float(thr)
    return best_thr, best_f1


threshold_grid = np.arange(0.05, 0.95, 0.05)
val_loader = data_module.val_dataloader()
val_probs = predict_proba_loader(model, val_loader)

best_threshold, best_val_f1 = pick_threshold_by_f1(y_val, val_probs, threshold_grid)

print("best_thr:", best_threshold)
print("best_val_f1:", best_val_f1)


best_thr: 0.3
best_val_f1: 0.6669927746695609


## Synthetic TEST metrics

In [45]:
def eval_binary(y_true: np.ndarray, probs: np.ndarray, threshold: float) -> Dict[str, float]:
    yhat = (probs >= threshold).astype(int)
    return {
        "acc": float(accuracy_score(y_true, yhat)),
        "prec": float(precision_score(y_true, yhat, zero_division=0)),
        "rec": float(recall_score(y_true, yhat, zero_division=0)),
        "f1": float(f1_score(y_true, yhat, zero_division=0)),
    }

test_loader = DataLoader(
    test_dataset,
    batch_size=4096,
    shuffle=False,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),
)
test_probs = predict_proba_loader(model, test_loader)

test_metrics = eval_binary(y_test, test_probs, best_threshold)
print("Synthetic TEST metrics:", test_metrics)

cm = confusion_matrix(y_test, (test_probs >= best_threshold).astype(int))
print("Confusion matrix [[tn, fp],[fn,tp]]:\n", cm)


Synthetic TEST metrics: {'acc': 0.5020104253193205, 'prec': 0.5010109570797264, 'rec': 0.9963282651353726, 'f1': 0.6667445429534716}
Confusion matrix [[tn, fp],[fn,tp]]:
 [[  639 82428]
 [  305 82762]]


# Transfer to tournament dataset

Now we evaluate **the same model** on a tournament dataset.

Tournament specifics (from your pipeline):
- embeddings are stored as **one NPZ** (because there is only one move column)
- labels are in `move_label` (0/1)
- recommended to filter by `is_used == True`

In [46]:
# === Load tournament CSV ===
tour_df = pd.read_csv(Path(TOUR_CSV_PATH))
print("Tournament rows:", len(tour_df))
print("is_used counts:", tour_df["is_used"].value_counts(dropna=False).to_dict())
print(tour_df[["player_elo", "move_label", "is_used"]].head())


Tournament rows: 77020
is_used counts: {False: 48610, True: 28410}
   player_elo  move_label  is_used
0        1883           0    False
1        1883           0    False
2        1883           0    False
3        1883           0    False
4        1883           0    False


In [47]:
# === Load tournament embeddings (NPZ) ===
with np.load(TOUR_EMB_NPZ) as tour_npz:
    tour_emb_all = tour_npz[TOUR_EMB_KEY]

is_used_tour = (tour_df["is_used"].to_numpy() == 1)
tour_row_idx = np.flatnonzero(is_used_tour)
tour_y = tour_df.loc[is_used_tour, "move_label"].to_numpy().astype(np.float32, copy=False)

class RowsWithLabelsDataset(Dataset):
    def __init__(self, emb: np.ndarray, row_idx: np.ndarray, y: np.ndarray):
        self.emb = emb
        self.row_idx = np.asarray(row_idx, dtype=np.int64)
        self.y = np.asarray(y, dtype=np.float32)

        if len(self.row_idx) != len(self.y):
            raise ValueError("row_idx and y must have the same length")

    def __len__(self) -> int:
        return int(len(self.row_idx))

    def __getitem__(self, idx: int):
        row = int(self.row_idx[idx])
        return self.emb[row], self.y[idx]


tour_dataset = RowsWithLabelsDataset(tour_emb_all, tour_row_idx, tour_y)

print("Tournament eligible:", len(tour_dataset), "rows")
print("Label distribution:", {0: int((tour_y==0).sum()), 1: int((tour_y==1).sum())})


Tournament eligible: 28410 rows
Label distribution: {0: 20235, 1: 8175}


## Tournament metrics

We reuse the threshold selected on synthetic validation.

In [48]:
tour_loader = DataLoader(
    tour_dataset,
    batch_size=4096,
    shuffle=False,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),
)
tour_probs = predict_proba_loader(model, tour_loader)

tour_metrics = eval_binary(tour_y, tour_probs, best_threshold)
print("Tournament metrics:", tour_metrics)

cm_tour = confusion_matrix(tour_y, (tour_probs >= best_threshold).astype(int))
print("Confusion matrix [[tn, fp],[fn,tp]]:\n", cm_tour)


Tournament metrics: {'acc': 0.2962689193945794, 'prec': 0.28950555713878595, 'rec': 0.9941284403669725, 'f1': 0.44842331779181727}
Confusion matrix [[tn, fp],[fn,tp]]:
 [[  290 19945]
 [   48  8127]]
